# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alikadirguzel/flyrankinternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Lane 2 — Refresh / Content Opportunity Scoring. This notebook writes the warehouse contract in plain words, checks three facts on the mid-panel month `2026-03`, builds five features that are knowable at the decision moment, then springs the leakage trap from notebook 02 on real warehouse rows.

The `_sample` table is June 2026 — the last month of the panel. It is sealed. All queries below hit `fact_content_daily_performance/month=2026-03` only.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-data-contracts` plus `flyrank/flyrank-data`.

## 0. Connect (token never lives in a cell)

Colab: put a Hugging Face **Read** token in Secrets as `HF_TOKEN`. Locally: set the same name as an environment variable. Request access first at [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse).

In [1]:
import os, sys, getpass, subprocess

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "pandas", "scikit-learn"]
)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from pathlib import Path
    for env_path in (Path(".env"), Path("../.env"), Path("../../.env")):
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line.startswith("HF_TOKEN=") and not line.startswith("#"):
                    HF_TOKEN = line.split("=", 1)[1].strip().strip('"').strip("'")
                    break
        if HF_TOKEN:
            break
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    if sys.stdin.isatty():
        HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
    else:
        raise RuntimeError(
            "No HF_TOKEN found. Colab: Secrets panel, name HF_TOKEN. "
            "Local: put HF_TOKEN=... in a gitignored .env file. Never paste it into a cell."
        )

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
MARCH_PARQUET = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("connected")
print("lane: Refresh / Content Opportunity Scoring")
print(f"iteration month: {MONTH}  (not the June _sample)")
print("tables this contract uses:")
print("  - fact_content_daily_performance  (month=2026-03 partition only)")
print("  - dim_clients                     (history start dates, access flags)")
print("  - dim_content                     (page dimension; join keys this week)")
print("deliberately not scanned: fact_content_query_90d, fact_content_daily_performance_sample")

march_cols = con.sql(f"DESCRIBE SELECT * FROM {MARCH_PARQUET}").df()
col_names = set(march_cols["column_name"].astype(str))
keep = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]
for extra in ("gsc_data_available", "ga4_data_available"):
    if extra in col_names:
        keep.append(extra)

print("caching March 2026 (needed columns only, one network pull)...")
con.execute(f"CREATE OR REPLACE TEMP TABLE march AS SELECT {', '.join(keep)} FROM {MARCH_PARQUET}")
con.execute(f"CREATE OR REPLACE TEMP TABLE dim_clients AS SELECT * FROM {DIM_CLIENTS}")
n_content = con.sql(f"SELECT COUNT(*) FROM {DIM_CONTENT}").fetchone()[0]
MARCH = "march"
DIM_CLIENTS = "dim_clients"

print(f"cached march rows: {con.sql('SELECT COUNT(*) FROM march').fetchone()[0]:,}")
print(f"dim_clients rows:  {con.sql('SELECT COUNT(*) FROM dim_clients').fetchone()[0]:,}")
print(f"dim_content rows:  {n_content:,}  (count only; not pulled into the feature frame)")


connected
lane: Refresh / Content Opportunity Scoring
iteration month: 2026-03  (not the June _sample)
tables this contract uses:
  - fact_content_daily_performance  (month=2026-03 partition only)
  - dim_clients                     (history start dates, access flags)
  - dim_content                     (page dimension; join keys this week)
deliberately not scanned: fact_content_query_90d, fact_content_daily_performance_sample


caching March 2026 (needed columns only, one network pull)...


cached march rows: 9,841,378
dim_clients rows:  104
dim_content rows:  519,606  (count only; not pulled into the feature frame)


## 1. Unit of analysis + time window

Five contract answers, in plain words. (Label and the excluded field are spelled out again in section 2.)

1. **One row means:** one published content item for one client, observed on one report date — `report_date × client_hash_id × content_hash_id`. That is the warehouse grain I query. The *decision* grain I will model later is coarser: one page (`client_hash_id`, `content_hash_id`) at a mid-month review.

2. **Tables:** `fact_content_daily_performance` (March 2026 partition) for the page-day river, `dim_clients` for history start dates, `dim_content` as the page dimension (join keys; not a feature source this week). Not the query table. Not the June `_sample`.

3. **Time window:** calendar month `2026-03` (`report_date` from 1 March through 31 March 2026). Decision moment = the midpoint of that month. Features come from the first half (already observed). The proxy label comes from the second half (not yet observed at the decision). June 2026 stays sealed.

4. **What I would predict / rank:** a refresh-priority score. The proxy I can compute inside this month: did Search Console impressions in the second half drop more than 20% versus the first half? (`imp_last_half < 0.8 × impressions_first_half`, with a small-volume floor). That is a current-window movement bucket, not a proof that a refresh will recover traffic.

5. **Deliberately excluded:** `fact_content_query_90d`. Its fixed 90-day window is the last ~90 days of the snapshot (ending 2026-06-30), so those query-mix columns sit in April–June — after a March decision. Using them here would be future information.

In [2]:
print("unit of analysis (warehouse grain): one row = report_date x client_hash_id x content_hash_id")
print("decision grain (later model row):   one page = client_hash_id x content_hash_id")
print("feature window: first half of 2026-03 (knowable at the mid-month decision)")
print("label window:   second half of 2026-03 (outcome after that decision)")
print("sealed:         June 2026 / _sample")

print("\nMarch columns loaded into the local temp table:")
print(march_cols[["column_name", "column_type"]].to_string(index=False))
print("\ncolumns actually cached for this contract:", ", ".join(keep))


unit of analysis (warehouse grain): one row = report_date x client_hash_id x content_hash_id
decision grain (later model row):   one page = client_hash_id x content_hash_id
feature window: first half of 2026-03 (knowable at the mid-month decision)
label window:   second half of 2026-03 (outcome after that decision)
sealed:         June 2026 / _sample

March columns loaded into the local temp table:
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_to

## 2. Fields: feature / label / context / excluded

Every field I touch sits in exactly one bucket.

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `impressions_first_half`, `clicks_first_half`, `avg_position_first_half`, `ctr_first_half`, `n_days_with_impressions` | Aggregated only from `report_date` on or before the mid-month decision. |
| **Label / proxy** | `is_declining_label`, and the ingredients it is computed from: `imp_last_half`, the last-half / first-half ratio | Second-half impressions are the outcome. They are never a feature. |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date`, `gsc_data_start`, `ga4_data_start`, `access_profile` | Join, group, split, read. Hashed ids are not model inputs. |
| **Excluded** | `fact_content_query_90d` (`impressions_90d`, `*_last30`, query hashes); June 2026 `_sample`; GA4 metrics on rows where `ga4_data_available` is not `TRUE` | Query table window is after March. June is the test month. GA4 zeros before tracking starts are not "no engagement". |

`= FALSE` and `NOT ga4_data_available` mishandle NULLs. Availability is three-valued. Filters below use `IS TRUE`.

In [3]:
buckets = pd.DataFrame(
    [
        ("feature", "impressions_first_half", "GSC impressions, first half of March"),
        ("feature", "clicks_first_half", "GSC clicks, first half of March"),
        ("feature", "avg_position_first_half", "mean gsc_avg_position where position > 0, first half"),
        ("feature", "ctr_first_half", "clicks / impressions, first half"),
        ("feature", "n_days_with_impressions", "days with impressions > 0, first half"),
        ("label", "is_declining_label", "1 if second-half impressions < 80% of first-half (floor applied)"),
        ("label", "imp_last_half", "ingredient of the label — never a feature"),
        ("context", "client_hash_id / content_hash_id", "join, grouped split — not a feature"),
        ("excluded", "fact_content_query_90d", "window sits after a March decision"),
        ("excluded", "_sample / June 2026", "sealed test month"),
    ],
    columns=["bucket", "field", "note"],
)
display(buckets)
print("availability rule: filter with IS TRUE / IS NOT TRUE, never = TRUE / = FALSE")


,bucket,field,note
0,feature,impressions_first_half,"GSC impressions, first half of March"
1,feature,clicks_first_half,"GSC clicks, first half of March"
2,feature,avg_position_first_half,"mean gsc_avg_position where position > 0, firs..."
3,feature,ctr_first_half,"clicks / impressions, first half"
4,feature,n_days_with_impressions,"days with impressions > 0, first half"
5,label,is_declining_label,1 if second-half impressions < 80% of first-ha...
6,label,imp_last_half,ingredient of the label — never a feature
7,context,client_hash_id / content_hash_id,"join, grouped split — not a feature"
8,excluded,fact_content_query_90d,window sits after a March decision
9,excluded,_sample / June 2026,sealed test month


availability rule: filter with IS TRUE / IS NOT TRUE, never = TRUE / = FALSE


## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries on `month=2026-03`. A contract line without a query next to it is a guess.

Then: five features from the same month, each with an "available when?" line, then the leakage trap.

### Query 1 — grain

If one row really is `report_date × client × content`, grouping on those three columns and asking for duplicates returns nothing.

In [4]:
grain = con.sql(f'''
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MARCH}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
''').df()

print("duplicate grain keys (expect 0 rows):", len(grain))
if len(grain) == 0:
    print("grain holds: one row is one content item, for one client, on one report_date")
else:
    display(grain)


duplicate grain keys (expect 0 rows): 0
grain holds: one row is one content item, for one client, on one report_date


### Query 2 — slice size and date span

In [5]:
span = con.sql(f'''
    SELECT
        COUNT(*)                    AS n_rows,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_pages,
        MIN(report_date)            AS min_date,
        MAX(report_date)            AS max_date
    FROM {MARCH}
''').df()

display(span)
print(
    f"March 2026 slice: {int(span.n_rows.iloc[0]):,} daily rows, "
    f"{int(span.n_pages.iloc[0]):,} pages, "
    f"{int(span.n_clients.iloc[0]):,} clients, "
    f"{span.min_date.iloc[0]} → {span.max_date.iloc[0]}"
)


,n_rows,n_clients,n_pages,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


March 2026 slice: 9,841,378 daily rows, 331,437 pages, 55 clients, 2026-03-01 00:00:00 → 2026-03-31 00:00:00


### Query 3 — availability (`IS TRUE`)

`ga4_data_available` (and its GSC twin) are three-valued: TRUE, FALSE, NULL. `= TRUE` drops NULLs the wrong way in some engines and is easy to misread. `IS TRUE` keeps only rows where the flag is actually true.

In [6]:
col_names = set(march_cols["column_name"].astype(str))
gsc_flag = "gsc_data_available" if "gsc_data_available" in col_names else None
ga4_flag = "ga4_data_available" if "ga4_data_available" in col_names else None
print("availability flags present:", [f for f in (gsc_flag, ga4_flag) if f])

selects = ["COUNT(*) AS n_rows"]
if gsc_flag:
    selects += [
        f"COUNT(*) FILTER (WHERE {gsc_flag} IS TRUE) AS gsc_is_true",
        f"COUNT(*) FILTER (WHERE {gsc_flag} IS NOT TRUE) AS gsc_not_true",
    ]
if ga4_flag:
    selects += [
        f"COUNT(*) FILTER (WHERE {ga4_flag} IS TRUE) AS ga4_is_true",
        f"COUNT(*) FILTER (WHERE {ga4_flag} IS NOT TRUE) AS ga4_not_true",
    ]
avail = con.sql(f"SELECT {', '.join(selects)} FROM {MARCH}").df()
display(avail)

n = int(avail.n_rows.iloc[0])
if "ga4_is_true" in avail.columns:
    ga4_ok = int(avail.ga4_is_true.iloc[0])
    print(f"GA4 usable rows (ga4_data_available IS TRUE): {ga4_ok:,} / {n:,}  ({ga4_ok / n:.1%})")
    print("the rest are FALSE or NULL — zeros there are not 'no engagement'")
if "gsc_is_true" in avail.columns:
    gsc_ok = int(avail.gsc_is_true.iloc[0])
    print(f"GSC usable rows (gsc_data_available IS TRUE): {gsc_ok:,} / {n:,}  ({gsc_ok / n:.1%})")


availability flags present: ['gsc_data_available', 'ga4_data_available']


,n_rows,gsc_is_true,gsc_not_true,ga4_is_true,ga4_not_true
0,9841378,3611061,6230317,413966,9427412


GA4 usable rows (ga4_data_available IS TRUE): 413,966 / 9,841,378  (4.2%)
the rest are FALSE or NULL — zeros there are not 'no engagement'
GSC usable rows (gsc_data_available IS TRUE): 3,611,061 / 9,841,378  (36.7%)


### Five features (max) — March 2026, knowable at the decision moment

Decision moment = midpoint of March. Each feature is aggregated from days **on or before** that midpoint.

| Feature | Available when? |
|---|---|
| `impressions_first_half` | knowable at the decision moment because those Search Console impressions already accrued in the first half of March |
| `clicks_first_half` | knowable at the decision moment because those clicks were already reported for the same days |
| `avg_position_first_half` | knowable at the decision moment because GSC had already published those ranks (`0` / missing is treated as no measurement, not rank zero) |
| `ctr_first_half` | knowable at the decision moment because it is clicks ÷ impressions from that same first half |
| `n_days_with_impressions` | knowable at the decision moment because it only counts first-half days that already had impressions |

The label column is built from the **second** half and is not in this feature list.

In [7]:
gsc_where = f"WHERE {gsc_flag} IS TRUE" if gsc_flag else ""

features = con.sql(f'''
    WITH bounds AS (
        SELECT
            MIN(report_date) AS min_d,
            MAX(report_date) AS max_d,
            MIN(report_date)
                + CAST((MAX(report_date) - MIN(report_date)) / 2 AS INTEGER)
                AS mid_d
        FROM {MARCH}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            ANY_VALUE(b.mid_d) AS decision_date,
            SUM(CASE WHEN f.report_date <= b.mid_d THEN f.gsc_impressions ELSE 0 END)
                AS impressions_first_half,
            SUM(CASE WHEN f.report_date <= b.mid_d THEN f.gsc_clicks ELSE 0 END)
                AS clicks_first_half,
            AVG(CASE
                    WHEN f.report_date <= b.mid_d AND f.gsc_avg_position > 0
                    THEN f.gsc_avg_position
                END) AS avg_position_first_half,
            COUNT(CASE
                    WHEN f.report_date <= b.mid_d AND f.gsc_impressions > 0
                    THEN 1
                END) AS n_days_with_impressions,
            SUM(CASE WHEN f.report_date > b.mid_d THEN f.gsc_impressions ELSE 0 END)
                AS imp_last_half
        FROM {MARCH} f, bounds b
        {gsc_where}
        GROUP BY 1, 2
        HAVING impressions_first_half >= 10
    )
    SELECT
        client_hash_id,
        content_hash_id,
        decision_date,
        impressions_first_half,
        clicks_first_half,
        avg_position_first_half,
        clicks_first_half / NULLIF(impressions_first_half, 0) AS ctr_first_half,
        n_days_with_impressions,
        imp_last_half,
        (imp_last_half < 0.8 * impressions_first_half)::INTEGER AS is_declining_label
    FROM windowed
''').df()

FEATURE_COLS = [
    "impressions_first_half",
    "clicks_first_half",
    "avg_position_first_half",
    "ctr_first_half",
    "n_days_with_impressions",
]

print(f"decision date used: {features['decision_date'].iloc[0]}")
print(f"feature-frame rows (pages with >=10 first-half impressions): {len(features):,}")
print(f"clients: {features['client_hash_id'].nunique():,}   pages: {features['content_hash_id'].nunique():,}")
print(f"proxy positive rate (is_declining_label=1): {features['is_declining_label'].mean():.1%}")
print()
print("available when?")
print("  impressions_first_half   — knowable at the decision moment because those impressions already accrued")
print("  clicks_first_half        — knowable at the decision moment because those clicks were already reported")
print("  avg_position_first_half  — knowable at the decision moment because GSC had already published those ranks")
print("  ctr_first_half           — knowable at the decision moment because it is first-half clicks / impressions")
print("  n_days_with_impressions  — knowable at the decision moment because it counts first-half days already observed")
print()
display(features[["client_hash_id", "content_hash_id"] + FEATURE_COLS + ["is_declining_label"]].head(8))


decision date used: 2026-03-16 00:00:00
feature-frame rows (pages with >=10 first-half impressions): 122,476
clients: 41   pages: 122,476
proxy positive rate (is_declining_label=1): 37.6%

available when?
  impressions_first_half   — knowable at the decision moment because those impressions already accrued
  clicks_first_half        — knowable at the decision moment because those clicks were already reported
  avg_position_first_half  — knowable at the decision moment because GSC had already published those ranks
  ctr_first_half           — knowable at the decision moment because it is first-half clicks / impressions
  n_days_with_impressions  — knowable at the decision moment because it counts first-half days already observed



,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,ctr_first_half,n_days_with_impressions,is_declining_label
0,client_62f4a7e64f5e0096,content_b5daab2b25cc1f1b,524.0,0.0,3.647711,0.000000,16,1
1,client_62f4a7e64f5e0096,content_5ed8e77f48b3a3f8,2785.0,9.0,6.651794,0.003232,16,1
2,client_62f4a7e64f5e0096,content_ff0c8b306ba1890a,45.0,0.0,16.965385,0.000000,15,1
3,client_62f4a7e64f5e0096,content_8975828df4457d75,2730.0,14.0,2.329559,0.005128,16,1
4,client_62f4a7e64f5e0096,content_7d1e1e869eb9fcad,206.0,1.0,11.454277,0.004854,16,1
5,client_62f4a7e64f5e0096,content_1ea05ae410dc43ec,808.0,4.0,12.174972,0.004950,16,0
6,client_62f4a7e64f5e0096,content_1cac8e0fc73b1e47,8513.0,6.0,5.818507,0.000705,16,1
7,client_62f4a7e64f5e0096,content_60589b03dad153cb,4686.0,6.0,4.063262,0.001280,16,0


### The trap — add a label-derived column, watch the score jump, then delete it

Notebook 02 fed `trend_pct` into a tree whose label was `trend_direction == "down"` and the tree simply relearned the rule. Same trick here: `imp_last_half` is an ingredient of `is_declining_label`. If I sneak it into the feature list, a depth-2 tree can reconstruct the proxy and the quick score races toward perfect. That number is worthless. I then drop the column and keep the honest score.

In [8]:
def precision_at_k(scores, y, k=50):
    y = np.asarray(y)
    scores = np.asarray(scores)
    top = np.argsort(-scores)[:k]
    return float(y[top].mean())


def quick_score(X, y, names, tag):
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    tree = DecisionTreeClassifier(
        max_depth=2, class_weight="balanced", random_state=42
    ).fit(X_tr, y_tr)
    proba = tree.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, proba)
    p50 = precision_at_k(proba, y_te, 50)
    print(f"{tag}")
    print(f"  ROC-AUC: {auc:.3f}    Precision@50: {p50:.3f}")
    print(export_text(tree, feature_names=list(names)))
    return auc, p50


y = features["is_declining_label"]
X_honest = features[FEATURE_COLS]
honest_auc, honest_p50 = quick_score(X_honest, y, FEATURE_COLS, "HONEST — five first-half features only")

X_leaky = features[FEATURE_COLS + ["imp_last_half"]]
leaky_auc, leaky_p50 = quick_score(
    X_leaky, y, FEATURE_COLS + ["imp_last_half"],
    "TRAP — same five features PLUS imp_last_half (label ingredient)",
)

print("trap sprung: the tree split on the future half and the score jumped toward perfect")
print(f"leaky  AUC {leaky_auc:.3f}  vs honest AUC {honest_auc:.3f}")
print(f"leaky  P@50 {leaky_p50:.3f} vs honest P@50 {honest_p50:.3f}")
print()
print("deleting imp_last_half from the feature list — keeping the honest number")
print(f"HONEST SCORE TO KEEP  ROC-AUC={honest_auc:.3f}  Precision@50={honest_p50:.3f}")
print("imp_last_half stays out. The label is derived from it; a feature must not be.")


HONEST — five first-half features only
  ROC-AUC: 0.588    Precision@50: 0.480
|--- ctr_first_half <= 0.00
|   |--- n_days_with_impressions <= 14.50
|   |   |--- class: 0
|   |--- n_days_with_impressions >  14.50
|   |   |--- class: 1
|--- ctr_first_half >  0.00
|   |--- clicks_first_half <= 4.50
|   |   |--- class: 0
|   |--- clicks_first_half >  4.50
|   |   |--- class: 0



TRAP — same five features PLUS imp_last_half (label ingredient)
  ROC-AUC: 0.695    Precision@50: 0.980
|--- imp_last_half <= 16.50
|   |--- imp_last_half <= 9.50
|   |   |--- class: 1
|   |--- imp_last_half >  9.50
|   |   |--- class: 1
|--- imp_last_half >  16.50
|   |--- n_days_with_impressions <= 14.50
|   |   |--- class: 0
|   |--- n_days_with_impressions >  14.50
|   |   |--- class: 1

trap sprung: the tree split on the future half and the score jumped toward perfect
leaky  AUC 0.695  vs honest AUC 0.588
leaky  P@50 0.980 vs honest P@50 0.480

deleting imp_last_half from the feature list — keeping the honest number
HONEST SCORE TO KEEP  ROC-AUC=0.588  Precision@50=0.480
imp_last_half stays out. The label is derived from it; a feature must not be.


## 4. Data limits

**Named limitation of this slice: unbalanced panel on a global March window.**

`dim_clients.gsc_data_start` is not the same day for every client. A single calendar month therefore does not mean "every client, full history." Clients who onboarded after 1 March 2026 never appear. Clients who started Search Console mid-panel look, in earlier months, as if they had no pages — that absence is missing tracking, not zero demand. GA4 is later still: rows before `ga4_data_start` are GSC-only, with `ga4_data_available` FALSE or NULL and zeros that are not "no engagement."

This notebook also cannot tell me whether a refresh *caused* a recovery. The proxy only marks observed impression movement inside March. June 2026 is unused on purpose, so I have not peeked at the natural future-outcome month.

In [9]:
panel = con.sql(f'''
    SELECT
        COUNT(*) AS n_clients,
        COUNT(gsc_data_start) AS with_gsc_start,
        COUNT(ga4_data_start) AS with_ga4_start,
        MIN(gsc_data_start) AS earliest_gsc,
        MAX(gsc_data_start) AS latest_gsc,
        COUNT(*) FILTER (
            WHERE gsc_data_start IS NULL OR gsc_data_start > make_date(2026, 3, 1)
        ) AS clients_not_started_by_march
    FROM {DIM_CLIENTS}
''').df()
display(panel)

print("limitation: a global March window is a policy, not a complete panel")
print("clients whose GSC history starts after 2026-03-01 are invisible in this slice")
print("I will not treat that invisibility as 'those clients had no content'.")


,n_clients,with_gsc_start,with_ga4_start,earliest_gsc,latest_gsc,clients_not_started_by_march
0,104,67,51,2025-01-27,2026-06-02,52


limitation: a global March window is a policy, not a complete panel
clients whose GSC history starts after 2026-03-01 are invisible in this slice
I will not treat that invisibility as 'those clients had no content'.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Contract recap:** one daily warehouse row is one page-day; I score pages from first-half March 2026 Search Console totals; I rank refresh review against a second-half impression-drop proxy; I exclude the query table and June; I checked grain, row span, and `IS TRUE` availability; I built five first-half features; I leaked `imp_last_half` on purpose, watched the score jump, and deleted it.